# Stellar Coordinate Explorer - Adding Galactic Coordinates (Random)

## Objective
Transform the ICRS (RA, Dec) coordinates of every star in filtered table of randomly-sampled sources to Galactic coordinates (l, b) and store them as new columns.

## Why this matters
- Galactic coordinates reveal how stars are distributed or structured relative to the Milky Way's plane in way that is not obvious in equitorial coordinates.
- Later visualizations (Aitoff projection) will use Galactic coordinates to show the whole sky.

## Dataset
- Input `bright_stars_filtered_random.fits` (saved from Day 3)
- Columns: `source_id`, `ra`, `dec`, `parallax`, `phot_g_mean_mag`, `bp_rp`
- Output: `stars_with_galactic_coord_random.fits`

## Goals for Today
- Load the 10000 sources table
- Create a `SkyCoord` object for all rows using RA and Dec (in degrees)
- Transform to Galactic Coordinates
- Add two new columns: `gal_l` (longitude) and `gal_b` (latitude)
- Verify that the transformed coordinates are within expected ranges ($l:\ 0-360^\degree$, $b:\ -90^\degree$ to $+90^\degree$)
- Save the enhanced table as `stars_with_galactic_coord_biased.fits`

## Checkpoint
- Table loaded successfully
- SkyCoord created without errors (watch for unit issues)
- New columns added and contain sensible numbers
- Quick Verification: print first 10 rows showing `RA`, `Dec`, `gal_l`, `gal_b`
- Enhanced table saved to disk

## Code
### Setup and Loading

In [15]:
# Setup
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

Now let's load the 10,000 sources to a Table 

In [16]:
table = Table.read("../../data/gaia_subset_random.fits")
table.info()

<Table length=10000>
      name       dtype  unit    class     n_bad
--------------- ------- ---- ------------ -----
      source_id   int64            Column     0
             ra float64  deg       Column     0
            dec float64  deg       Column     0
       parallax float64  mas       Column     0
phot_g_mean_mag float32  mag       Column     0
          bp_rp float32  mag MaskedColumn    12


### Obtaining the Equitorial Coordinates and Transforming to Galactic
We now create a SkyCoord object containing equitorial frame coordinates for the 10,000 sources.

In [17]:
eq_coord = SkyCoord(table["ra"], table["dec"], unit=(u.deg, u.deg), frame="icrs")

Checking the saved coordinates:

In [18]:
print(f"{'#' * 20} Confirm the coordinate components of the 10000 Gaia stars in the ICRS frame {'#' * 20}")
print(eq_coord)
eq_coord.ra, eq_coord.dec

#################### Confirm the coordinate components of the 10000 Gaia stars in the ICRS frame ####################
<SkyCoord (ICRS): (ra, dec) in deg
    [( 89.08891529, -23.37556436), ( 89.75122195,  71.33384612),
     ( 37.93089459,  19.29079672), ..., (161.75628355, -58.57703175),
     (147.99151894, -17.34748735), (356.56865606,  -6.69885104)]>


(<Longitude [ 89.08891529,  89.75122195,  37.93089459, ..., 161.75628355,
             147.99151894, 356.56865606] deg>,
 <Latitude [-23.37556436,  71.33384612,  19.29079672, ..., -58.57703175,
            -17.34748735,  -6.69885104] deg>)

Transforming these coordinates to the Galactic frame:

In [19]:
gal_coord = eq_coord.galactic
print(f"{'#' * 20} Confirm the coordinate components of the 10000 Gaia stars in the ICRS frame {'#' * 20}")
print(gal_coord)
gal_coord.b, gal_coord.l

#################### Confirm the coordinate components of the 10000 Gaia stars in the ICRS frame ####################
<SkyCoord (Galactic): (l, b) in deg
    [(228.80311412, -22.04747484), (142.5137847 ,  21.55527558),
     (153.25352095, -37.60631071), ..., (287.30755554,   0.46878662),
     (253.36839325,  27.78242782), ( 82.83403133, -64.37105615)]>


(<Latitude [-22.04747484,  21.55527558, -37.60631071, ...,   0.46878662,
             27.78242782, -64.37105615] deg>,
 <Longitude [228.80311412, 142.5137847 , 153.25352095, ..., 287.30755554,
             253.36839325,  82.83403133] deg>)

Now we can confirm whether these coordinates are within the expected ranges in the Galactic frame ($l:\ 0-360^\degree$, $b:\ -90^\degree$ to $+90^\degree$).

In [20]:
print(f"l range: {gal_coord.l.min()} to {gal_coord.l.max()}")
print(f"b range: {gal_coord.b.min()} to {gal_coord.b.max()}")

l range: 0.032137670218243 deg to 359.99016990296735 deg
b range: -88.46956199476621 deg to 88.96055716000978 deg


Let's now save the new Galactic Coordinates in our table.

In [21]:
table.add_columns((gal_coord.l, gal_coord.b), names=('gal_l', 'gal_b'), indexes=(3, 3))
print(f"{'#' * 20} Confirm the Galactic Coordinates for the Gaia stars {'#' * 20}")
print(table.info())

#################### Confirm the Galactic Coordinates for the Gaia stars ####################
<Table length=10000>
      name       dtype  unit    class     n_bad
--------------- ------- ---- ------------ -----
      source_id   int64            Column     0
             ra float64  deg       Column     0
            dec float64  deg       Column     0
          gal_l float64  deg       Column     0
          gal_b float64  deg       Column     0
       parallax float64  mas       Column     0
phot_g_mean_mag float32  mag       Column     0
          bp_rp float32  mag MaskedColumn    12
None


In [22]:
print(f"{'#' * 20} View the first 10 sources {'#' * 20}")
table[:10]

#################### View the first 10 sources ####################


source_id,ra,dec,gal_l,gal_b,parallax,phot_g_mean_mag,bp_rp
,deg,deg,deg,deg,mas,mag,mag
int64,float64,float64,float64,float64,float64,float32,float32
2916442949023646464,89.08891529461751,-23.375564356338113,228.803114121309,-22.047474836107785,6.174716638660832,9.528021,0.71763134
486087738386602624,89.75122195274709,71.33384612083127,142.51378469524832,21.555275575541973,6.786864259555966,8.923034,0.5152149
86268270726618624,37.93089459029714,19.29079672106284,153.2535209526649,-37.606310710681264,5.567776813866466,8.103913,1.2267585
6245995261529926784,245.12519193816152,-18.658756095050595,356.6954940323096,21.725614735614677,12.422733521538417,9.504698,1.0528879
4599710073355218048,259.3530395685651,30.4113148729394,53.30360558588356,32.47659162857363,7.205622033945202,9.934753,0.8441076
6839283420817505280,330.3870919481059,-15.61204705231511,40.327453588239635,-49.021334557207744,7.0114632292318735,6.828051,1.0603137
6140009865392980224,192.90687913704932,-41.0327107343903,302.9704462113673,21.839022669266384,8.776928834986256,9.370744,0.7011404
3096062707589083136,120.38431429991603,6.27166537770808,215.3036282356422,18.433362552611314,5.16299083464015,9.515524,0.62593746


Now we save our table to a file named `stars_with_galactic_biased.fits`.

In [23]:
table.write('../../data/stars_with_galactic_coord_random.fits')

## Basic Observations

The range in galactic coordinates is approximately 0 to 360 in the galactic longitude and approximately -88 to 89 in the galactic latitude. This is expected for a randomly selected set of sources in contrast with the biased selection for the earlier set of data in previous notebooks.